In [2]:
!pip install torch

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
    --------------------------------------- 2.9/123.0 MB 20.1 MB/s eta 0:00:06
   -- ------------------------------------- 6.3/123.0 MB 18.6 MB/s eta 0:00:07
   --- ------------------------------------ 10.5/123.0 MB 21.7 MB/s eta 0:00:06
   ---- ----------------------------------- 12.6/123.0 MB 16.6 MB/s eta 0:00:07
   ---- ----------------------------------- 13.9/123.0 MB 14.6 MB/s eta 0:00:08
   ---- ----------------------------------- 14.9/123.0 MB 12.6 MB/s eta 0:00:09
   ----- ---------------------------------- 16.0/123.0 MB 11.4 MB/s eta 0:00:10
   ----- ---------------------------------- 17.0/123.0 MB 10.5 MB/s eta 0:00:11
   ----- ---------------------------------- 18.1/123.0 MB 9.9 MB/s eta 0:00:11
   ------ --------------------------------- 19.1/123.0 MB 9.5 MB/s eta 0:00:11
   -----


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import numpy as np
import joblib

In [5]:
X_train = np.load(
    "../data/processed/X_train.npy"
)

X_test = np.load(
    "../data/processed/X_test.npy"
)

y_train = np.load(
    "../data/processed/y_train.npy"
)

y_test = np.load(
    "../data/processed/y_test.npy"
)

In [6]:
label_encoder = joblib.load(
    "../data/processed/label_encoder.pkl"
)

In [7]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(189, 2151)
(48, 2151)
(189,)
(48,)


In [8]:
import torch

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

print(X_train_tensor.shape)
print(y_train_tensor.shape)

torch.Size([189, 2151])
torch.Size([189])


In [9]:
X_train_tensor = X_train_tensor.unsqueeze(1)

X_test_tensor = X_test_tensor.unsqueeze(1)

print(X_train_tensor.shape)
print(X_test_tensor.shape)

torch.Size([189, 1, 2151])
torch.Size([48, 1, 2151])


In [10]:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Test batches :", len(test_loader))

Train batches: 12
Test batches : 3


In [11]:
import torch.nn as nn

class MineralCNN(nn.Module):

    def __init__(self, num_classes=38):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv1d(
                in_channels=1,
                out_channels=16,
                kernel_size=5,
                padding=2
            ),

            nn.ReLU(),

            nn.MaxPool1d(2),

            nn.Conv1d(
                16,
                32,
                kernel_size=5,
                padding=2
            ),

            nn.ReLU(),

            nn.MaxPool1d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                32 * 537,
                256
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                256,
                num_classes
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

In [12]:
model = MineralCNN(
    num_classes=38
)

print(model)

MineralCNN(
  (features): Sequential(
    (0): Conv1d(1, 16, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(16, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): ReLU()
    (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=17184, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=38, bias=True)
  )
)


In [13]:
sample_batch = next(
    iter(train_loader)
)

X_batch, y_batch = sample_batch

outputs = model(X_batch)

print(outputs.shape)

torch.Size([16, 38])


In [14]:
print(X_train_tensor.shape)
print(X_test_tensor.shape)

print(len(train_loader))
print(len(test_loader))

print(outputs.shape)

torch.Size([189, 1, 2151])
torch.Size([48, 1, 2151])
12
3
torch.Size([16, 38])
